# NAGARNETRA — ANPR Model Fine-Tuning
### Fine-tune YOLOv8 on Indian License Plates

**Team: The Underthinker | Smart India Hackathon 2026**

This notebook fine-tunes YOLOv8 to detect Indian license plate bounding boxes.
After training, copy `runs/detect/train/weights/best.pt` to `models/best.pt`.

---

## Dataset

Use the **Indian License Plates with Labels** dataset from Roboflow:
- URL: https://universe.roboflow.com/sih-qqlqc/indian-license-plates-with-labels
- Download in **YOLOv8 format** (requires free Roboflow account)
- Extract to `datasets/indian-plates/`

Expected structure:
```
datasets/indian-plates/
  data.yaml
  train/
    images/  labels/
  valid/
    images/  labels/
  test/
    images/  labels/
```

In [ ]:
# Install dependencies
!pip install ultralytics==8.2.28 torch torchvision --quiet

In [ ]:
from ultralytics import YOLO
import yaml
from pathlib import Path

# Verify dataset path
DATASET_PATH = Path('datasets/indian-plates/data.yaml')
assert DATASET_PATH.exists(), (
    f'Dataset not found at {DATASET_PATH}.\n'
    'Download from: https://universe.roboflow.com/sih-qqlqc/indian-license-plates-with-labels\n'
    'Select YOLOv8 format and extract to datasets/indian-plates/'
)

with DATASET_PATH.open() as f:
    ds_config = yaml.safe_load(f)
print('Dataset config:', ds_config)

In [ ]:
# Load base model — yolov8n is fastest, yolov8m for better accuracy
# Model size tradeoff: n (fastest) < s < m < l < x (most accurate)
MODEL_SIZE = 'n'  # change to 'm' for better accuracy at cost of speed

model = YOLO(f'yolov8{MODEL_SIZE}.pt')
print(f'Loaded YOLOv8{MODEL_SIZE} base model')

In [ ]:
# Fine-tuning configuration
EPOCHS = 50           # increase to 100 for higher accuracy
IMG_SIZE = 640        # standard YOLO input size
BATCH = 16            # reduce to 8 if OOM error
DEVICE = 'cpu'        # change to '0' for GPU (CUDA), or 'mps' for Apple Silicon

results = model.train(
    data=str(DATASET_PATH),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    name='nagarnetra_anpr',
    project='runs/detect',
    # Augmentation — helps with Indian street conditions
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,        # small rotation for slanted cameras
    shear=2.0,
    perspective=0.0005,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
)

print('Training complete!')
print(f'Best weights: runs/detect/nagarnetra_anpr/weights/best.pt')

In [ ]:
# Validate on test set
best_model = YOLO('runs/detect/nagarnetra_anpr/weights/best.pt')
metrics = best_model.val(data=str(DATASET_PATH), split='test')
print(f'mAP@0.5: {metrics.box.map50:.3f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.3f}')

In [ ]:
import shutil
from pathlib import Path

# Copy best weights to models/ directory
src = Path('runs/detect/nagarnetra_anpr/weights/best.pt')
dst = Path('../models/best.pt')
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(src, dst)
print(f'✅ Copied {src} → {dst}')
print('Restart NAGARNETRA backend to use the fine-tuned model.')
print('Set YOLO_WEIGHTS_PATH=./models/best.pt in .env (this is already the default).')

## After Training

1. `models/best.pt` is now your fine-tuned ANPR detector
2. Restart the backend: `docker compose restart backend`
3. Re-run ingestion: `python scripts/ingest_videos.py`
4. Accuracy should now be significantly higher on Indian plates

## Tips for Higher Accuracy

- **More data**: Augment with photos of Indian plates from multiple states
- **Night footage**: Add nighttime/low-light samples
- **More epochs**: Try 100–200 epochs
- **Larger model**: Switch to `yolov8m` or `yolov8l`
- **Two-stage**: Fine-tune separately for detection (bounding box) and recognition (OCR) if PaddleOCR accuracy is the bottleneck